# 多目标规划：一个模型一个 Notebook

## 这个模型解决什么问题
两个目标互相冲突时（利润要高、排放要低；成本要低、覆盖率要高），
不存在"一个最优解"，只有一组**互不支配**的解，叫 Pareto 前沿。

## 两种常用处理方式

**① 加权和**：`min w1·f1 + w2·f2`，扫不同权重得到一组解。
简单，但权重怎么定要说清楚，而且非凸前沿上的点可能取不到。

**② ε-约束**：固定一个目标，把另一个目标变成 `f2 <= ε` 的约束，扫 ε。
能取到非凸部分的点，是论文里更稳妥的写法。

## 什么叫"互不支配"
解 A 支配解 B，意思是 A 在两个目标上都不比 B 差、至少一个更好。
Pareto 前沿上的点，就是谁也支配不了谁的那批解——它们都是"同等合理"的方案，
最终选哪个要给出偏好理由（比如"排放优先"或"利润优先"）。

## 目录
0 环境 / 1 配置 / 2 数据接口 / 3 求解（加权和 + ε-约束）/ 4 示例 / 5 导出

> 依赖 `numpy pandas scipy`；目标与约束都必须是线性的（非线性多目标不在本 Notebook 范围）。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    tag="多目标规划",
    n_points=8,          # ε 扫描的点数：越多前沿越细，但算得越久
    make_plots=True,
    save_excel=True,
)
print("配置：", CFG)


配置： {'tag': '多目标规划', 'n_points': 8, 'make_plots': True, 'save_excel': True}


## 2. 数据接口

多目标的输入通常就是系数矩阵，直接构造即可；如果来自原始表，先用 `数据学习.ipynb` 或上面几个 Notebook 的做法整理成矩阵。


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 求解：加权和 + ε-约束


In [4]:
# =====================================================================
# 3. 求解：加权和 + ε-约束
# =====================================================================
def solve_weighted(objectives, A_ub=None, b_ub=None, bounds=None,
                   maximize=None, weights=None):
    """加权和法：min Σ w_k · (方向调整后的 f_k)。

    objectives : (k, n) 系数矩阵，每行一个目标
    maximize   : 每个目标的方向，bool 或 list
    weights    : 权重向量；None 则用均匀权重
    """
    Obj = np.atleast_2d(np.asarray(objectives, dtype=float))
    k, n = Obj.shape
    maximize = ([True] * k if maximize is None
                else [maximize] * k if isinstance(maximize, bool) else list(maximize))
    w = np.ones(k) / k if weights is None else np.asarray(weights, dtype=float)
    # 最大化目标取负，统一成最小化
    c = sum(w[i] * (-Obj[i] if maximize[i] else Obj[i]) for i in range(k))
    r = sp_opt.linprog(c, A_ub=A_ub, b_ub=b_ub,
                       bounds=bounds or [(0, None)] * n, method="highs")
    return r


def solve_epsilon(objectives, A_ub, b_ub, bounds, maximize, n_points=8):
    """ε-约束法（两个目标时）：优化目标1，把目标2 压成 <= ε 的约束，扫 ε。"""
    Obj = np.atleast_2d(np.asarray(objectives, dtype=float))
    if Obj.shape[0] != 2:
        raise ValueError("ε-约束这里只实现两个目标；3 个以上请用加权和")
    mx1, mx2 = maximize
    c1 = -Obj[0] if mx1 else Obj[0]

    def solve_with_eps(eps):
        row = -Obj[1].reshape(1, -1) if mx2 else Obj[1].reshape(1, -1)
        bnd = np.array([-eps if mx2 else eps])
        A = np.vstack([A_ub, row]) if A_ub is not None else row
        b = np.concatenate([b_ub, bnd]) if b_ub is not None else bnd
        return sp_opt.linprog(c1, A_ub=A, b_ub=b, bounds=bounds, method="highs")

    # 先求两个单目标最优，得到目标2 的取值区间
    r1 = sp_opt.linprog(c1, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    r2 = sp_opt.linprog(-Obj[1] if mx2 else Obj[1], A_ub=A_ub, b_ub=b_ub,
                        bounds=bounds, method="highs")
    if r1.x is None or r2.x is None:
        return pd.DataFrame(), []
    f2_a, f2_b = float(Obj[1] @ r1.x), float(Obj[1] @ r2.x)
    lo, hi = min(f2_a, f2_b), max(f2_a, f2_b)

    rows = []
    for eps in np.linspace(lo, hi, n_points):
        r = solve_with_eps(eps)
        if r.x is None:
            continue
        rows.append({"目标1": round(float(Obj[0] @ r.x), 3),
                     "目标2": round(float(Obj[1] @ r.x), 3),
                     **{f"x{i+1}": round(float(v), 3) for i, v in enumerate(r.x)}})
    return pd.DataFrame(rows), [lo, hi]


def weighted_sweep(objectives, A_ub, b_ub, bounds, maximize, steps=6):
    """扫一组权重，看解怎么随偏好移动（论文里做灵敏度分析很直观）。"""
    Obj = np.atleast_2d(np.asarray(objectives, dtype=float))
    rows = []
    for a in np.linspace(0, 1, steps):
        w = [a, 1 - a]
        r = solve_weighted(Obj, A_ub, b_ub, bounds, maximize, w)
        if r.x is None:
            continue
        rows.append({"w1": round(a, 2), "w2": round(1 - a, 2),
                     "目标1": round(float(Obj[0] @ r.x), 3),
                     "目标2": round(float(Obj[1] @ r.x), 3),
                     **{f"x{i+1}": round(float(v), 3) for i, v in enumerate(r.x)}})
    return pd.DataFrame(rows)


## 4. 示例：利润 vs 原料消耗


In [5]:
# =====================================================================
# 4. 示例：生产计划的「利润 vs 原料消耗」双目标
# =====================================================================
# 三个产品：利润系数 520/380/640；资源消耗只留两条约束方便二维画图。
obj_matrix = np.array([
    [520, 380, 640],      # 目标1：总利润（最大化）
    [3.5, 2.0, 4.5],      # 目标2：原料消耗（最小化）
], dtype=float)
A_ub = [[2.0, 1.5, 3.0],  # 设备工时
        [1.2, 0.8, 1.8]]  # 人工
b_ub = [800, 500]
bounds = [(0, None)] * 3
maximize = [True, False]

print("目标方向：利润最大化、原料消耗最小化")
print("约束：设备工时 <= 800，人工 <= 500\n")

# ---- 加权和：看偏好怎么移动解 ----
t_weight = weighted_sweep(obj_matrix, A_ub, b_ub, bounds, maximize, steps=6)
print("=== 加权和扫描 ===")
print(t_weight.to_string(index=False))

# ---- ε-约束：得到 Pareto 前沿 ----
t_front, rng = solve_epsilon(obj_matrix, A_ub, b_ub, bounds, maximize, CFG["n_points"])
print(f"\n=== ε-约束 Pareto 前沿（目标2 区间 {rng[0]:.2f} ~ {rng[1]:.2f}）===")
print(t_front.to_string(index=False))

# 非支配筛选：去掉被其他点支配的解
def non_dominated(df, c1="目标1", c2="目标2"):
    keep = []
    for i, row in df.iterrows():
        dominated = ((df[c1] >= row[c1]) & (df[c2] <= row[c2]) &
                     ((df[c1] > row[c1]) | (df[c2] < row[c2]))).any()
        if not dominated:
            keep.append(i)
    return df.loc[keep].sort_values(c1, ascending=False).reset_index(drop=True)

front_clean = non_dominated(t_front)
print(f"\n去重后的非支配解：{len(front_clean)} 个")
print(front_clean.to_string(index=False))

# 选一个"推荐方案"：这里用归一化后离理想点最近的解（TOPSIS 思路）
if len(front_clean):
    f1 = front_clean["目标1"]; f2 = front_clean["目标2"]
    n1 = (f1 - f1.min()) / max(f1.max() - f1.min(), 1e-9)
    n2 = (f2 - f2.min()) / max(f2.max() - f2.min(), 1e-9)
    pick = (n1 + (1 - n2)).idxmax()
    print("\n推荐方案（离理想点最近）：")
    print(front_clean.loc[[pick]].to_string(index=False))


目标方向：利润最大化、原料消耗最小化
约束：设备工时 <= 800，人工 <= 500

=== 加权和扫描 ===
 w1  w2      目标1    目标2    x1  x2  x3
0.0 1.0      0.0    0.0   0.0 0.0 0.0
0.2 0.8 208000.0 1400.0 400.0 0.0 0.0
0.4 0.6 208000.0 1400.0 400.0 0.0 0.0
0.6 0.4 208000.0 1400.0 400.0 0.0 0.0
0.8 0.2 208000.0 1400.0 400.0 0.0 0.0
1.0 0.0 208000.0 1400.0 400.0 0.0 0.0

=== ε-约束 Pareto 前沿（目标2 区间 0.00 ~ 1400.00）===
     目标1    目标2    x1    x2  x3
     0.0    0.0   0.0  -0.0 0.0
 38000.0  200.0   0.0 100.0 0.0
 76000.0  400.0   0.0 200.0 0.0
114000.0  600.0   0.0 300.0 0.0
152000.0  800.0   0.0 400.0 0.0
190000.0 1000.0   0.0 500.0 0.0
204800.0 1200.0 160.0 320.0 0.0
208000.0 1400.0 400.0  -0.0 0.0

去重后的非支配解：8 个
     目标1    目标2    x1    x2  x3
208000.0 1400.0 400.0  -0.0 0.0
204800.0 1200.0 160.0 320.0 0.0
190000.0 1000.0   0.0 500.0 0.0
152000.0  800.0   0.0 400.0 0.0
114000.0  600.0   0.0 300.0 0.0
 76000.0  400.0   0.0 200.0 0.0
 38000.0  200.0   0.0 100.0 0.0
     0.0    0.0   0.0  -0.0 0.0

推荐方案（离理想点最近）：
     目标1    目标2  x1    x

## 5. 结果导出


In [6]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [7]:
# =====================================================================
# 5. 结果导出与 Pareto 前沿图
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL and len(front_clean) > 1:
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    ax.scatter(t_front["目标1"], t_front["目标2"], c="#c9ccd1", label="ε 扫描全部解")
    ax.scatter(front_clean["目标1"], front_clean["目标2"], c="#4a9d6a",
               zorder=3, label="非支配解")
    ax.plot(front_clean["目标1"], front_clean["目标2"], ls="--", c="#4a9d6a", alpha=.6)
    ax.set_xlabel("目标1：总利润（越大越好）")
    ax.set_ylabel("目标2：原料消耗（越小越好）")
    ax.set_title("Pareto 前沿：两目标此消彼长")
    ax.legend(fontsize=9)
    fig.tight_layout()

written = export_result(
    {"Pareto前沿": t_front, "非支配解": front_clean, "加权和扫描": t_weight},
    CFG, fig=fig, fig_name="Pareto前沿")
print(written)


[表格] 多目标规划_结果汇总.xlsx（3 个 Sheet）
[图片] 多目标规划_Pareto前沿.png
[输出]  D:\git项目\输出数据\优化_多目标规划
{'主表': WindowsPath('D:/git项目/输出数据/优化_多目标规划/多目标规划_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_多目标规划/多目标规划_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_多目标规划/多目标规划_Pareto前沿.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。

**写论文时**：Pareto 前沿只是「候选方案集」，必须单独用一段说明最终方案怎么选（偏好、权重、理想点距离），否则评委会问「凭什么选这个」。
